# DeepSniffing on Colab: one run from an empty runtime

Runs the whole project on a **Colab Pro L4**: clone, Python 3.13, data build, the three training steps,
the evaluations, and a zip of everything at the end, downloaded straight through the browser.

**Before you start**

1. Runtime ▸ Change runtime type ▸ **L4 GPU**.
2. Add `GH_TOKEN` under the key icon in the left bar (Colab ▸ Secrets), with a GitHub token that can read
   this private repository. It is used for the clone and for the refresh, and is never written into this
   notebook or into `.git/config`.
3. Expect **three hours or so**: the data build is the slow part, then the pretraining.

Nothing is saved between sessions. The last cell zips the checkpoints, the evaluations, the report and this
notebook, and downloads the zip to your machine. Do not close the tab while a cell runs.

**One rule.** Edit this notebook locally, commit it, then launch the runtime and run it top to bottom without
editing cells. Colab executes the copy you opened, which is not necessarily the copy in the clone. The cell
after the clone compares the two and warns if they differ.

**What the runtime can see.** The clone fetches `main` and nothing else. A commit that lives on `dev`, or on
any other branch, does not reach Colab: merge it into `main` before a run.


## 1. Clone the repository

The clone takes `main`, the branch a Colab session can see. Anything that is still on `dev` will not be here.

The repository is private, so the clone is authenticated with a token from Colab Secrets. The remote is reset
to the plain URL straight afterwards, so the token is not left behind in `.git/config`.

If the clone already exists, it is fetched and reset to `main` instead, so a rerun in the same session picks
up whatever was pushed since.


In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import userdata
REPO = Path("/content/DeepSniffing")
URL = "https://github.com/suprch4rg3d/DeepSniffing.git"
BRANCH = "main"
try:
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
auth = f"https://{token}@github.com/suprch4rg3d/DeepSniffing.git" if token else URL
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, auth, str(REPO)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", URL], cwd=REPO, check=True)
    print("cloned", BRANCH)
else:
    subprocess.run(["git", "fetch", "--depth", "1", auth, BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO, check=True)
    print("refreshed", BRANCH)
os.chdir(REPO)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", os.getcwd(), "| branch:", BRANCH, "| HEAD:", head)

### Self-check: is the running notebook the one in the clone?

If you edited cells before launching, the code that executes may not be the code in the repository. This
compares the two and warns on any difference. It is best effort: a notebook opened straight from GitHub is
not on the VM's disk, and then there is nothing to compare against.


In [ ]:
import json
from pathlib import Path
running = Path("/content/notebooks/deepsniffing-on-colab.ipynb")
committed = Path("/content/DeepSniffing/notebooks/deepsniffing-on-colab.ipynb")
def code_cells(path):
    nb = json.loads(Path(path).read_text())
    return ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
if not running.is_file():
    print("no copy of this notebook on the VM to compare; skipping the check")
elif code_cells(running) == code_cells(committed):
    print("OK - the running notebook matches the clone")
else:
    print("WARNING: the running notebook differs from the clone")
    print("  cells here:", len(code_cells(running)), "| cells in the clone:", len(code_cells(committed)))
    print("  restart the runtime and run the notebook that is in the repository")

## 2. Python 3.13 and the project

Colab ships Python 3.12 and `pyproject.toml` asks for 3.13, so `uv` fetches an interpreter and makes a venv.

The project is then installed by Poetry, from `poetry.lock`, so every package gets the version the report was
measured with: torch 2.14.1, rdkit and pandas and the rest. Installing with `pip install -e .` instead would
read the ranges in `pyproject.toml` and quietly take whatever is newest that day.


In [ ]:
!pip install -q uv
VENV = "/content/dsn-venv"
!uv venv --python 3.13 {VENV}
!uv pip install --python {VENV}/bin/python poetry
!cd /content/DeepSniffing && POETRY_VIRTUALENVS_CREATE=false {VENV}/bin/poetry install --only main
print("environment ready at", VENV)


## 3. Check the accelerator

The cell raises an error and stops when there is no GPU. Without one the pretraining would run for days, and
the session would die long before it finished.


In [ ]:
!{VENV}/bin/python -c "import torch; assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > L4 GPU'; p = torch.cuda.get_device_properties(0); print(f'{p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}')"

## 4. Build the data

`ds-prepare` downloads the Pyrfume odor sources at a pinned revision, a few MB.

`ds-build` downloads PubChem's `CID-SMILES.gz` and `CID-Mass.gz` (2.8 GB together), unpacks them (about 14 GB
on disk), keeps the compounds of 350 Da or less, converts 3.4 million of them to canonical SMILES and SELFIES,
and writes the tables, the split, the vocabulary and the packed tokens. This is the slow step, **30 to 60
minutes**. The download happens before any table is read, so a rebuilt session pays it again; there is nothing
cheap to cache short of the 17 GB of raw archives.


In [ ]:
import shutil
free = shutil.disk_usage("/content").free / 1e9
print(f"{free:.0f} GB free on /content")
if free < 20:
    raise RuntimeError(f"{free:.0f} GB free; ds-build needs about 20 GB")

In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/ds-prepare

In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/ds-build

## 5. Train the three steps

Each step keeps its own folder under `output/odor/`, saves after every epoch and resumes if the same cell is
run again, within the same session.

- **step 1, pretraining**: 24 epochs over a set of 100 million tokens. The slowest, **about two hours on an L4**.
- **step 2, generator**: 160 epochs over the 6,344 odor molecules plus replay, **ten to twenty minutes**.
- **step 3, classifier**: a head on the frozen decoder, **a couple of minutes**.


In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/ds-train --stage 1

In [ ]:
import glob
import os
import zipfile
with zipfile.ZipFile("/content/stage1-snapshot.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for path in glob.glob("/content/DeepSniffing/output/odor/stage1/*"):
        zf.write(path, os.path.relpath(path, "/content/DeepSniffing"))
print("snapshot:", os.path.getsize("/content/stage1-snapshot.zip"), "bytes")
try:
    from google.colab import files as colab_files
    colab_files.download("/content/stage1-snapshot.zip")
except ImportError:
    pass

In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/ds-train --stage 2

In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/ds-train --stage 3

## 6. Measure

The generator is scored on the three held-out sets, then the classifier, then `report` builds `results.md` and
the figures from the JSON the two evaluations wrote.

This is not quick either: each split writes 1,400 molecules, seven requests of two hundred, and scores every
held-out molecule three times over exact probabilities. Expect **ten to twenty minutes** for the three
generator splits, a few minutes for the classifier, and a minute for the report.


In [ ]:
for split in ("validation", "test", "dream"):
    print("=" * 20, "generator", split)
    !cd /content/DeepSniffing && {VENV}/bin/python -m scripts.core.evaluate --split {split}

In [ ]:
for split in ("validation", "test", "dream"):
    print("=" * 20, "classifier", split)
    !cd /content/DeepSniffing && {VENV}/bin/python -m scripts.core.classifier --split {split}

In [ ]:
!cd /content/DeepSniffing && {VENV}/bin/python -m scripts.core.report

## 7. Collect everything and download

Zips the three checkpoints, every evaluation JSON, the three training histories, the report and this notebook,
and writes a `RUN.txt` with the commit, the date, the GPU and the library versions that produced them. Then
hands the zip to the browser. That download is the reliable path out of the VM; a mounted Drive does not
always show fresh files on other devices.


In [ ]:
import glob
import os
import subprocess
import zipfile
from datetime import UTC, datetime
from pathlib import Path
root = Path("/content/DeepSniffing")
out = root / "output" / "odor"
def package_version(name):
    r = subprocess.run([f"{VENV}/bin/python", "-c", f"from importlib.metadata import version; print(version('{name}'))"], capture_output=True, text=True)
    return (r.stdout or r.stderr).strip().splitlines()[-1] if (r.stdout or r.stderr).strip() else "unknown"
def gpu_name():
    r = subprocess.run([f"{VENV}/bin/python", "-c", "import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"], capture_output=True, text=True)
    return r.stdout.strip()
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=root, capture_output=True, text=True).stdout.strip()
manifest = "\n".join([
    f"commit: {commit}",
    f"date: {datetime.now(UTC).isoformat(timespec='seconds')}",
    f"gpu: {gpu_name()}",
    f"torch: {package_version('torch')}",
    f"rdkit: {package_version('rdkit')}",
    f"selfies: {package_version('selfies')}",
])
patterns = [
    "stage1/best.pt", "stage2/best.pt", "stage3/best.pt",
    "evaluation_*.json", "stage3/evaluation_*.json",
    "stage1/history.csv", "stage2/history.csv", "stage3/history.csv",
    "report/results.md", "report/*.png",
]
wanted = []
for pattern in patterns:
    wanted += sorted(glob.glob(str(out / pattern)))
notebook = root / "notebooks" / "deepsniffing-on-colab.ipynb"
if notebook.exists():
    wanted.insert(0, str(notebook))
zip_name = "/content/deepsniffing-colab-run.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("RUN.txt", manifest)
    for path in wanted:
        zf.write(path, os.path.relpath(path, root))
        print(f"{os.path.getsize(path)/1e6:8.2f} MB  {os.path.relpath(path, root)}")
print()
print(manifest)
print("wrote", zip_name, f"({os.path.getsize(zip_name)/1e6:.1f} MB)")
missing = [p for p in ("stage2/best.pt", "stage3/best.pt", "evaluation_test.json") if not (out / p).exists()]
if missing:
    print("MISSING:", missing, "- the zip is incomplete; do not commit it")
try:
    from google.colab import files as colab_files
    colab_files.download(zip_name)
    print("download triggered; check the browser Downloads folder")
except ImportError:
    print("not a Colab runtime; the zip is at", zip_name)

## 8. Put the artefacts in the repository

Unzip on your machine into the clone, then commit. The checkpoints are about 25 MB each. Keep `RUN.txt` too:
it is the only record of which commit and which GPU produced them.

```bash
cd ~/Documents/GitHub/DeepSniffing
unzip ~/Downloads/deepsniffing-colab-run.zip
git status --short
git add output/odor/stage1/best.pt output/odor/stage2/best.pt output/odor/stage3/best.pt
git add output/odor/evaluation_*.json output/odor/stage3/evaluation_*.json RUN.txt
git commit -m "Add the models and evaluations of the Colab run"
git push origin <your-branch>
```

Then merge that branch into `main`. Colab only ever clones `main`, so a commit on `dev` is invisible to the
next session.

The report and the figures stay out of the commit unless you want them; they are rebuilt by
`python -m scripts.core.report` from the JSONs.
